# Notebook 01: Foundation Models

Before we can adapt a model, we need to be able to load one and talk to it. This notebook does that, and explains the main ideas you will keep bumping into: what a foundation model is, how it reads text, and what "parameters" means.

## What is a foundation model?

A foundation model is a very large network that has been trained on an enormous pile of text: web pages, books, code, articles. The training is simple to describe. The model is shown text with the next word hidden, and it tries to guess that word, over and over, billions of times. Do that at a big enough scale and the model ends up picking up grammar, facts, and a surprising amount of reasoning along the way.

Some names you may have heard:

- **GPT-4** (OpenAI): closed. You can only use it through their service, not download it.
- **Llama 3** (Meta): open. You can download it and run it yourself.
- **Phi-2 and Phi-3** (Microsoft): small open models that punch above their weight.
- **Mistral**: open, and strong for its size.

We use **TinyLlama** (1.1 billion parameters) for most of this project. It is small enough to run on a Mac, but it works in exactly the same way as the giant models, so everything you learn here carries straight over.

## How does it actually read text?

Modern language models are built on a design called the **Transformer** (introduced by Google in 2017). You do not need the maths, but two ideas are worth knowing:

- **Tokens.** The model does not read letters or whole words. It reads "tokens", which are word-pieces. For example "financial" might become "fin" plus "ancial". Splitting words this way lets the model cope with words it has never seen by building them from familiar pieces.
- **Attention.** Older models read strictly left to right. A Transformer can look at every word at once and work out which words matter most to each other. That is what lets it keep track of meaning across a long sentence.

The model is built from many of these layers stacked up, and at the very end it does one thing: predict the next token. That is all it ever does. Generating a whole answer is just doing that prediction again and again.

## What are parameters?

The model's **parameters** (also called weights) are the numbers that hold everything it has learned. TinyLlama has 1.1 billion of them. GPT-4 is estimated to have over a trillion.

Fine-tuning means adjusting these numbers. The catch is that adjusting all 1.1 billion is slow and memory-hungry. In notebook 04 we use a trick called LoRA that adjusts only a tiny slice of them, which is what makes training possible on a laptop.


In [ ]:
# -- Imports ------------------------------------------------------------------
# transformers: Hugging Face's library for loading and running pre-trained models
# torch: PyTorch - the deep learning framework everything is built on

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Detect device (MPS = Apple Silicon, cuda = NVIDIA, cpu = fallback)
device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

## 1. Loading a model from Hugging Face

Hugging Face is a website that hosts thousands of free, ready-to-use models. Think of it like an app store for AI models. We can pull one down with a couple of lines of code.

We load two pieces:

- The **tokeniser**, which turns text into the number form the model needs (and back again).
- The **model** itself, which is the network that does the thinking.

The first time you run the next cell it downloads TinyLlama (around 600 MB), so it may take a minute. After that it is saved on your machine and loads quickly.


In [ ]:
# We'll use TinyLlama - a 1.1B parameter model that's fast to load and run.
# It was trained on 3 trillion tokens of text, same architecture as Llama 2.
# Model card: https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokeniser...")
tokeniser = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # Use 16-bit floats to halve memory usage
    device_map="auto",            # Automatically place layers on available device
)

print("\nModel loaded successfully!")
print(f"Model type   : {type(model).__name__}")

## 2. A look at the tokeniser

The model cannot read letters. Everything has to become numbers first, and that is the tokeniser's job.

As mentioned, tokens are usually word-pieces rather than whole words. A common word like "the" is one token, but a longer or rarer word gets split into a few. The next cell takes a financial sentence and shows you exactly how it gets chopped up and numbered, so you can see what the model actually receives.


In [ ]:
# Let's see how the tokeniser breaks down a financial sentence

example_text = "The company's revenue grew 14% year-over-year driven by strong EPS."

# Encode: text -> token IDs
token_ids = tokeniser.encode(example_text)

# Convert IDs back to human-readable tokens (not full words - sub-word pieces)
tokens = tokeniser.convert_ids_to_tokens(token_ids)

print("Original text:")
print(f"  {example_text}\n")

print("Tokens (sub-word pieces):")
print(f"  {tokens}\n")

print("Token IDs (what the model actually sees):")
print(f"  {token_ids}\n")

print(f"Character count : {len(example_text)}")
print(f"Token count     : {len(token_ids)}")
print(f"\nNote: models have a 'context window' - a maximum number of tokens they can process at once.")
print(f"TinyLlama's context window: 2048 tokens")

## 3. Looking inside the model

Now we peek at how the model is built. This is not just for curiosity. In notebook 04 we add the LoRA training trick to specific parts of the model, so it helps to see those parts first and know what we are pointing at.


In [ ]:
# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters    : {total_params:,}  ({total_params/1e9:.2f}B)")
print(f"Trainable parameters: {trainable_params:,}")
print()

# Print the high-level layer structure
print("Model architecture (top level):")
for name, module in model.named_children():
    print(f"  {name}: {type(module).__name__}")

print()
print("A transformer is a stack of identical 'decoder layers'.")
print("Each layer contains: self-attention + feed-forward network + layer normalisation.")

In [ ]:
# Zoom into one transformer layer to see its components

print("Components inside a single transformer layer (layer 0):")
for name, module in model.model.layers[0].named_modules():
    if name:  # skip the root
        print(f"  {name}: {type(module).__name__}")

### What you are looking at

Each layer of the model is made of a few repeating parts:

- `self_attn` is the attention mechanism, the part that decides which words to focus on. Inside it are four matrices named `q_proj`, `k_proj`, `v_proj`, and `o_proj`.
- `mlp` is the feed-forward network. After attention has decided what is relevant, this part does the actual reshaping of each word's representation.
- `input_layernorm` and `post_attention_layernorm` are there to keep the numbers stable during training. You can think of them as housekeeping.

The names to remember are `q_proj` and `v_proj`. When we do LoRA in notebook 04, we attach small trainable add-ons to exactly those two, and leave everything else untouched. That is the whole idea behind training cheaply.

## 4. Running the model and seeing what it knows


In [ ]:
# Let's ask the base model a financial question.
# This shows us the starting point BEFORE any adaptation.
# We'll compare this to the fine-tuned model's answers in Notebook 06.

def generate_response(prompt, max_new_tokens=200, temperature=0.7):
    """
    Run inference on the model.
    
    - prompt: the input text
    - max_new_tokens: maximum tokens to generate
    - temperature: controls randomness. Lower = more deterministic, higher = more creative.
    """
    # Format as a chat message (this model expects a specific template)
    messages = [
        {"role": "system", "content": "You are a helpful financial analyst."},
        {"role": "user", "content": prompt},
    ]
    
    # Apply the chat template - converts the messages dict into the format the model expects
    formatted = tokeniser.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenise and move to device
    inputs = tokeniser(formatted, return_tensors="pt").to(model.device)
    
    # Generate tokens
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokeniser.eos_token_id,
        )
    
    # Decode: strip the input tokens, return only the generated response
    response = tokeniser.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response


# Test 1: General financial question (model should handle this fine)
question = "What is the difference between revenue and profit?"
print(f"Question: {question}")
print("-" * 60)
response = generate_response(question)
print(response)

In [ ]:
# Test 2: A question requiring specific document knowledge
# The model will likely hallucinate or be vague here - it has no access to specific filings.
# This is the problem RAG and fine-tuning solve.

question = "What was Apple's total revenue for fiscal year 2023, and what drove growth?"
print(f"Question: {question}")
print("-" * 60)
print("Base model response (no retrieval, no fine-tuning):")
response = generate_response(question)
print(response)
print()
print("Note: the model may give plausible but vague or incorrect figures.")
print("This is exactly the hallucination problem we address in later notebooks.")

## 5. How much memory does a model need?

Model size matters because it decides whether something will even run on your machine. The next cell does a rough calculation for a few well-known models so you can see why we picked a small one.

The rule of thumb: in the common "float16" format, each parameter takes about 2 bytes of memory. So a 1.1 billion parameter model needs a couple of gigabytes, while a 70 billion one needs well over a hundred. That is the difference between a laptop and a server rack.


In [ ]:
# Understanding how much memory models use is important for choosing what to run.
# A rough rule: each parameter in float16 (half precision) uses 2 bytes.

def estimate_model_memory(num_params_billions, precision="float16"):
    bytes_per_param = {"float32": 4, "float16": 2, "int8": 1, "int4": 0.5}
    memory_gb = num_params_billions * 1e9 * bytes_per_param[precision] / (1024**3)
    return memory_gb

models = {
    "TinyLlama (1.1B)": 1.1,
    "Phi-2 (2.7B)": 2.7,
    "Mistral (7B)": 7,
    "Llama-3 (8B)": 8,
    "Llama-3 (70B)": 70,
}

print(f"{'Model':<25} {'float32':>12} {'float16':>12} {'int4 (quantised)':>18}")
print("-" * 70)
for name, params in models.items():
    f32 = estimate_model_memory(params, "float32")
    f16 = estimate_model_memory(params, "float16")
    i4  = estimate_model_memory(params, "int4")
    print(f"{name:<25} {f32:>10.1f}GB {f16:>10.1f}GB {i4:>16.1f}GB")

print()
print("MacBook Air M4 has 16GB or 32GB of unified memory (shared CPU+GPU).")
print("We target models that fit comfortably in float16 - TinyLlama and Phi-2 are ideal.")

## Recap

In this notebook we:

- Loaded a real open-source model from Hugging Face in just a couple of lines
- Saw how text gets split into tokens before the model can use it
- Looked inside the model at the attention and feed-forward parts, and noted where LoRA will go later
- Ran the model and watched it answer a general question well, then get vague on a specific one
- Worked out roughly how much memory different model sizes need

The big takeaway: the base model is smart in general but knows nothing about your specific documents. When we asked it about a particular company's figures, it had to guess. Fixing that is the whole point of the rest of the project, and there are two ways to do it: give the model the documents to read (RAG, notebook 03), or train it on them (fine-tuning, notebook 04).

Next up: notebook 02, Data Preparation.
